<a href="https://colab.research.google.com/github/maddygoodro/MIS433/blob/main/Vibe%20Coding%20Challenge%20meal_workout_planner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🥗 7-Day Meal Planner — TheMealDB API
Run each cell **in order**. Enter your health goals when prompted in Cell 3.

> Free API docs: https://www.themealdb.com/api.php

In [1]:
# Cell 1 — Install dependencies
!pip install requests pandas tabulate --quiet

In [2]:
# Cell 2 — Imports
import requests
import random
import pandas as pd
from tabulate import tabulate
from IPython.display import display, HTML

In [3]:
# Cell 3 — User health goal inputs
print("=" * 55)
print("        🥗 Personalised 7-Day Meal Planner")
print("=" * 55)
print("\nPlease enter your health goals below.\n")

name = input("Your name (optional, press Enter to skip): ").strip() or "Friend"

daily_calories  = int(input("Daily calorie target (e.g. 2000): ").strip())
daily_protein_g = int(input("Daily protein target in grams (e.g. 150): ").strip())

print("\nDietary preference:")
print("  1 = No restriction")
print("  2 = High protein (gym / muscle gain)")
print("  3 = Low calorie (weight loss)")
print("  4 = Vegetarian / plant-based")
print("  5 = Low carb / keto-friendly")
diet_choice = input("Enter number (1-5): ").strip()

diet_map = {
    "1": "balanced",
    "2": "high_protein",
    "3": "low_calorie",
    "4": "vegetarian",
    "5": "low_carb",
}
diet_label = diet_map.get(diet_choice, "balanced")

print("\nFitness / health goal:")
print("  1 = Lose weight")
print("  2 = Maintain weight")
print("  3 = Build muscle")
print("  4 = Improve energy levels")
goal_choice = input("Enter number (1-4): ").strip()

goal_map = {
    "1": "weight_loss",
    "2": "maintenance",
    "3": "muscle_gain",
    "4": "energy",
}
health_goal = goal_map.get(goal_choice, "maintenance")

print(f"\n✅ Got it, {name}! Building your plan…\n")

        🥗 Personalised 7-Day Meal Planner

Please enter your health goals below.

Your name (optional, press Enter to skip): Maddy
Daily calorie target (e.g. 2000): 2200
Daily protein target in grams (e.g. 150): 120

Dietary preference:
  1 = No restriction
  2 = High protein (gym / muscle gain)
  3 = Low calorie (weight loss)
  4 = Vegetarian / plant-based
  5 = Low carb / keto-friendly
Enter number (1-5): 3

Fitness / health goal:
  1 = Lose weight
  2 = Maintain weight
  3 = Build muscle
  4 = Improve energy levels
Enter number (1-4): 1

✅ Got it, Maddy! Building your plan…



In [4]:
# Cell 4 — TheMealDB API helpers
BASE_URL = "https://www.themealdb.com/api/json/v1/1"

def get_meal_by_category(category: str) -> dict | None:
    """Return one random meal from a TheMealDB category."""
    url  = f"{BASE_URL}/filter.php?c={category}"
    resp = requests.get(url, timeout=10)
    data = resp.json().get("meals")
    if not data:
        return None
    meal_stub  = random.choice(data)
    detail_url = f"{BASE_URL}/lookup.php?i={meal_stub['idMeal']}"
    detail     = requests.get(detail_url, timeout=10).json()
    meals      = detail.get("meals")
    return meals[0] if meals else None


def get_random_meal() -> dict | None:
    """Return a completely random meal."""
    url  = f"{BASE_URL}/random.php"
    data = requests.get(url, timeout=10).json().get("meals")
    return data[0] if data else None


def get_available_categories() -> list[str]:
    url  = f"{BASE_URL}/categories.php"
    data = requests.get(url, timeout=10).json().get("categories", [])
    return [c["strCategory"] for c in data]


def search_meals_by_name(query: str) -> list[dict]:
    url  = f"{BASE_URL}/search.php?s={query}"
    data = requests.get(url, timeout=10).json().get("meals") or []
    return data

In [5]:
# Cell 5 — Map diet / goal to category pools
ALL_CATEGORIES = get_available_categories()

CATEGORY_POOLS = {
    "high_protein": ["Chicken", "Beef", "Seafood", "Lamb", "Pork"],
    "weight_loss":  ["Chicken", "Seafood", "Vegetarian", "Vegan", "Side"],
    "muscle_gain":  ["Chicken", "Beef", "Seafood", "Lamb", "Pasta"],
    "energy":       ["Chicken", "Pasta", "Breakfast", "Miscellaneous", "Side"],
    "low_calorie":  ["Vegetarian", "Vegan", "Seafood", "Side", "Chicken"],
    "low_carb":     ["Chicken", "Beef", "Seafood", "Lamb", "Pork"],
    "vegetarian":   ["Vegetarian", "Vegan", "Pasta", "Side", "Miscellaneous"],
    "balanced":     ALL_CATEGORIES,
    "maintenance":  ALL_CATEGORIES,
}

if diet_label in CATEGORY_POOLS:
    pool = CATEGORY_POOLS[diet_label]
elif health_goal in CATEGORY_POOLS:
    pool = CATEGORY_POOLS[health_goal]
else:
    pool = ALL_CATEGORIES

pool = [c for c in pool if c in ALL_CATEGORIES] or ALL_CATEGORIES
print(f"🍽  Category pool for your plan: {', '.join(pool)}\n")

🍽  Category pool for your plan: Vegetarian, Vegan, Seafood, Side, Chicken



In [6]:
# Cell 6 — Nutrition estimator
# TheMealDB free tier does not provide macros — we estimate from category averages.

NUTRITION_ESTIMATES = {
    "Chicken":       {"cal": 450, "protein": 38, "carbs": 20, "fat": 14},
    "Beef":          {"cal": 520, "protein": 40, "carbs": 15, "fat": 22},
    "Seafood":       {"cal": 380, "protein": 35, "carbs": 18, "fat": 10},
    "Lamb":          {"cal": 490, "protein": 36, "carbs": 14, "fat": 24},
    "Pork":          {"cal": 460, "protein": 34, "carbs": 18, "fat": 20},
    "Vegetarian":    {"cal": 320, "protein": 12, "carbs": 40, "fat": 10},
    "Vegan":         {"cal": 290, "protein": 10, "carbs": 42, "fat":  8},
    "Pasta":         {"cal": 500, "protein": 18, "carbs": 68, "fat": 14},
    "Breakfast":     {"cal": 410, "protein": 16, "carbs": 45, "fat": 16},
    "Side":          {"cal": 220, "protein":  6, "carbs": 32, "fat":  8},
    "Dessert":       {"cal": 380, "protein":  5, "carbs": 55, "fat": 14},
    "Miscellaneous": {"cal": 400, "protein": 20, "carbs": 35, "fat": 15},
    "Starter":       {"cal": 280, "protein": 12, "carbs": 28, "fat": 10},
}
DEFAULT_NUTRITION = {"cal": 400, "protein": 20, "carbs": 35, "fat": 15}

def estimate_nutrition(category: str) -> dict:
    base = NUTRITION_ESTIMATES.get(category, DEFAULT_NUTRITION).copy()
    for k in base:
        base[k] = int(base[k] * random.uniform(0.85, 1.15))
    return base

In [7]:
# Cell 7 — Build 7-day meal plan
DAYS  = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
MEALS = ["Breakfast", "Lunch", "Dinner"]

meal_plan = {}
seen_ids  = set()

print("⏳ Fetching recipes from TheMealDB…\n")

for day in DAYS:
    meal_plan[day] = {}
    for slot in MEALS:
        if slot == "Breakfast" and "Breakfast" in pool:
            cat = "Breakfast"
        else:
            cat = random.choice(pool)

        meal     = None
        attempts = 0
        while attempts < 5:
            meal = get_meal_by_category(cat)
            if meal and meal["idMeal"] not in seen_ids:
                break
            cat = random.choice(pool)
            attempts += 1

        if meal is None:
            meal = get_random_meal()

        if meal:
            seen_ids.add(meal["idMeal"])
            nutrition = estimate_nutrition(meal.get("strCategory", cat))
            meal_plan[day][slot] = {
                "name":      meal["strMeal"],
                "category":  meal.get("strCategory", cat),
                "nutrition": nutrition,
                "url":       meal.get("strSource") or f"https://www.themealdb.com/meal/{meal['idMeal']}",
                "thumb":     meal.get("strMealThumb", ""),
                "id":        meal["idMeal"],
            }

print("✅ Recipes fetched!\n")

⏳ Fetching recipes from TheMealDB…

✅ Recipes fetched!



In [8]:
# Cell 8 — Display full meal plan as table
rows = []
for day in DAYS:
    for slot in MEALS:
        entry = meal_plan[day].get(slot, {})
        n     = entry.get("nutrition", {})
        rows.append({
            "Day":       day,
            "Meal":      slot,
            "Recipe":    entry.get("name", "N/A"),
            "Category":  entry.get("category", ""),
            "Cal":       n.get("cal", 0),
            "Protein g": n.get("protein", 0),
            "Carbs g":   n.get("carbs", 0),
            "Fat g":     n.get("fat", 0),
        })

df = pd.DataFrame(rows)
print(tabulate(df, headers="keys", tablefmt="rounded_outline", showindex=False))

╭───────────┬───────────┬──────────────────────────────────────┬────────────┬───────┬─────────────┬───────────┬─────────╮
│ Day       │ Meal      │ Recipe                               │ Category   │   Cal │   Protein g │   Carbs g │   Fat g │
├───────────┼───────────┼──────────────────────────────────────┼────────────┼───────┼─────────────┼───────────┼─────────┤
│ Monday    │ Breakfast │ Singapore Noodles with Shrimp        │ Seafood    │   424 │          40 │        18 │      11 │
│ Monday    │ Lunch     │ Chicken Fried Rice                   │ Chicken    │   428 │          43 │        17 │      12 │
│ Monday    │ Dinner    │ Roast fennel and aubergine paella    │ Vegan      │   295 │           8 │        38 │       8 │
│ Tuesday   │ Breakfast │ Eggplant Adobo                       │ Vegetarian │   324 │          11 │        37 │       9 │
│ Tuesday   │ Lunch     │ Thai rice noodle salad               │ Vegetarian │   287 │          10 │        36 │       8 │
│ Tuesday   │ Dinner    

In [9]:
# Cell 9 — Daily nutrition summary vs. goals
print("\n" + "=" * 60)
print(f"  📊 Daily Nutrition Summary  |  Target: {daily_calories} kcal / {daily_protein_g}g protein")
print("=" * 60)

summary_rows = []
for day in DAYS:
    day_cal  = sum(meal_plan[day][s]["nutrition"]["cal"]     for s in MEALS if s in meal_plan[day])
    day_prot = sum(meal_plan[day][s]["nutrition"]["protein"] for s in MEALS if s in meal_plan[day])
    day_carb = sum(meal_plan[day][s]["nutrition"]["carbs"]   for s in MEALS if s in meal_plan[day])
    day_fat  = sum(meal_plan[day][s]["nutrition"]["fat"]     for s in MEALS if s in meal_plan[day])

    cal_diff  = day_cal  - daily_calories
    prot_diff = day_prot - daily_protein_g

    cal_flag  = "✅" if abs(cal_diff)  <= daily_calories  * 0.10 else ("⬆️ " if cal_diff  > 0 else "⬇️ ")
    prot_flag = "✅" if abs(prot_diff) <= daily_protein_g * 0.15 else ("⬆️ " if prot_diff > 0 else "⬇️ ")

    summary_rows.append({
        "Day":       day,
        "Total Cal": day_cal,
        "Cal Goal":  f"{cal_flag} {cal_diff:+d}",
        "Protein g": day_prot,
        "Prot Goal": f"{prot_flag} {prot_diff:+d}g",
        "Carbs g":   day_carb,
        "Fat g":     day_fat,
    })

df_summary = pd.DataFrame(summary_rows)
print(tabulate(df_summary, headers="keys", tablefmt="rounded_outline", showindex=False))


  📊 Daily Nutrition Summary  |  Target: 2200 kcal / 120g protein
╭───────────┬─────────────┬────────────┬─────────────┬─────────────┬───────────┬─────────╮
│ Day       │   Total Cal │ Cal Goal   │   Protein g │ Prot Goal   │   Carbs g │   Fat g │
├───────────┼─────────────┼────────────┼─────────────┼─────────────┼───────────┼─────────┤
│ Monday    │        1147 │ ⬇️  -1053  │          91 │ ⬇️  -29g    │        73 │      31 │
│ Tuesday   │         804 │ ⬇️  -1396  │          27 │ ⬇️  -93g    │       109 │      24 │
│ Wednesday │         802 │ ⬇️  -1398  │          46 │ ⬇️  -74g    │        79 │      25 │
│ Thursday  │         906 │ ⬇️  -1294  │          52 │ ⬇️  -68g    │        95 │      26 │
│ Friday    │        1013 │ ⬇️  -1187  │          86 │ ⬇️  -34g    │        70 │      32 │
│ Saturday  │        1017 │ ⬇️  -1183  │          59 │ ⬇️  -61g    │       100 │      30 │
│ Sunday    │        1102 │ ⬇️  -1098  │          80 │ ⬇️  -40g    │        80 │      28 │
╰───────────┴───────────

In [12]:
# Cell 10 — Rich HTML recipe cards (renders inline in Colab)
html_parts = ["""
<style>
  body { font-family: 'Segoe UI', sans-serif; }
  h2   { color: #2d6a4f; }
  h3   { color: #40916c; margin-top: 24px; }
  .meal-card {
    display: inline-block; width: 280px; vertical-align: top;
    border: 1px solid #d8f3dc; border-radius: 12px;
    padding: 12px; margin: 8px; background: #f0faf3;
    box-shadow: 0 2px 6px rgba(0,0,0,.08);
  }
  .meal-card img  { width: 100%; border-radius: 8px; margin-bottom: 8px; }
  .meal-title     { font-weight: 700; font-size: 14px; color: #1b4332; }
  .meal-slot      { font-size: 11px; color: #74c69d; text-transform: uppercase; letter-spacing: 1px; }
  .meal-nutrition { font-size: 12px; color: #555; margin-top: 6px; }
  .meal-nutrition span { margin-right: 8px; }
  a               { color: #40916c; font-size: 12px; }
</style>
<h2>🥗 Your Personalised 7-Day Meal Plan</h2>
"""]

for day in DAYS:
    html_parts.append(f"<h3>📅 {day}</h3>")
    for slot in MEALS:
        entry   = meal_plan[day].get(slot, {})
        n       = entry.get("nutrition", {})
        thumb   = entry.get("thumb", "")
        url     = entry.get("url", "#")
        img_tag = f'<img src="{thumb}" alt="{entry.get("name"," Builders Inc.")}">' if thumb else ""
        html_parts.append(f"""
        <div class="meal-card">
          {img_tag}
          <div class="meal-slot">{slot}</div>
          <div class="meal-title">{entry.get("name","N/A")}</div>
          <div class="meal-nutrition">
            <span>🔥 {n.get("cal",0)} kcal</span>
            <span>💪 {n.get("protein",0)}g protein</span>
            <span>🍞 {n.get("carbs",0)}g carbs</span>
            <span>🧈 {n.get("fat",0)}g fat</span>
          </div>
          <a href="{url}" target="_blank">View recipe →</a>
        </div>""")

display(HTML("".join(html_parts)))

In [13]:
# Cell 11 — Export to CSV and download
output_path = "meal_plan_7day.csv"
df.to_csv(output_path, index=False)
print(f"\n💾 Meal plan saved to: {output_path}")

try:
    from google.colab import files
    files.download(output_path)
    print("⬇️  Download started in your browser.")
except ImportError:
    print("(Not running in Colab — file saved locally.)")


💾 Meal plan saved to: meal_plan_7day.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Download started in your browser.


---
## 🏋️ Weekly Workout Routine
The next cells use the **OpenAI API** to generate a personalised 7-day workout plan that complements your meal plan and health goals.

In [14]:
# Cell 12 — Install OpenAI SDK
!pip install openai --quiet

In [15]:
# Cell 13 — OpenAI API key setup
import os
from google.colab import userdata

# Reads your secret called OPENAI_API_KEY from Colab Secrets
# (Colab sidebar → 🔑 Secrets → add key: OPENAI_API_KEY)
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

from openai import OpenAI
client = OpenAI()
print("✅ OpenAI client ready.")

✅ OpenAI client ready.


In [16]:
# Cell 14 — Additional workout preference inputs
print("=" * 55)
print("        🏋️  Weekly Workout Planner")
print("=" * 55)
print("\nA few more questions to tailor your workout routine.\n")

print("Fitness level:")
print("  1 = Beginner (0-6 months experience)")
print("  2 = Intermediate (6 months - 2 years)")
print("  3 = Advanced (2+ years)")
fitness_level_choice = input("Enter number (1-3): ").strip()
fitness_level_map = {"1": "beginner", "2": "intermediate", "3": "advanced"}
fitness_level = fitness_level_map.get(fitness_level_choice, "beginner")

print("\nAvailable equipment:")
print("  1 = No equipment (bodyweight only)")
print("  2 = Dumbbells / resistance bands")
print("  3 = Full gym access")
equipment_choice = input("Enter number (1-3): ").strip()
equipment_map = {
    "1": "bodyweight only, no equipment",
    "2": "dumbbells and resistance bands",
    "3": "full gym with barbells, cables, machines",
}
equipment = equipment_map.get(equipment_choice, "bodyweight only")

workout_days = input("\nHow many days per week can you work out? (1-7): ").strip()
try:
    workout_days = max(1, min(7, int(workout_days)))
except ValueError:
    workout_days = 3

session_minutes = input("How many minutes per session? (e.g. 45): ").strip()
try:
    session_minutes = max(10, min(120, int(session_minutes)))
except ValueError:
    session_minutes = 45

injuries = input("\nAny injuries or areas to avoid? (press Enter to skip): ").strip() or "none"

print(f"\n✅ Got it! Generating your workout plan with OpenAI…\n")

        🏋️  Weekly Workout Planner

A few more questions to tailor your workout routine.

Fitness level:
  1 = Beginner (0-6 months experience)
  2 = Intermediate (6 months - 2 years)
  3 = Advanced (2+ years)
Enter number (1-3): 1

Available equipment:
  1 = No equipment (bodyweight only)
  2 = Dumbbells / resistance bands
  3 = Full gym access
Enter number (1-3): 3

How many days per week can you work out? (1-7): 3
How many minutes per session? (e.g. 45): 60

Any injuries or areas to avoid? (press Enter to skip): 

✅ Got it! Generating your workout plan with OpenAI…



In [17]:
# Cell 15 — Generate workout plan via OpenAI
import json as _json

goal_labels = {
    "weight_loss":  "lose weight and burn fat",
    "maintenance":  "maintain current fitness",
    "muscle_gain":  "build muscle and increase strength",
    "energy":       "improve energy and overall fitness",
}
goal_text = goal_labels.get(health_goal, "improve overall fitness")

prompt = f"""You are an expert personal trainer. Create a detailed 7-day weekly workout plan.

User profile:
- Health goal: {goal_text}
- Dietary preference: {diet_label.replace('_', ' ')}
- Daily calorie target: {daily_calories} kcal
- Daily protein target: {daily_protein_g}g
- Fitness level: {fitness_level}
- Equipment available: {equipment}
- Workout days per week: {workout_days}
- Session length: {session_minutes} minutes
- Injuries / areas to avoid: {injuries}

Return ONLY a valid JSON object with this exact structure (no markdown, no extra text):
{{
  "plan_summary": "2-3 sentence overview of the plan and its rationale",
  "days": {{
    "Monday":    {{"type": "Rest|<workout type>", "focus": "<muscle group or cardio focus>", "duration_min": <int>, "exercises": [{{"name": "<exercise>", "sets": <int>, "reps_or_duration": "<e.g. 12 reps or 30 sec>", "rest_sec": <int>, "tip": "<form tip>"}}]}},
    "Tuesday":   {{...}},
    "Wednesday": {{...}},
    "Thursday":  {{...}},
    "Friday":    {{...}},
    "Saturday":  {{...}},
    "Sunday":    {{...}}
  }}
}}
For rest days, set exercises to an empty list [].
Distribute the {workout_days} active days across the week with appropriate rest."""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.7,
    max_tokens=3000,
)

raw = response.choices[0].message.content.strip()
# Strip markdown fences if present
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

workout_plan = _json.loads(raw)
print("✅ Workout plan generated!\n")
print("📋 Plan summary:")
print(workout_plan.get("plan_summary", ""))

✅ Workout plan generated!

📋 Plan summary:
This 7-day workout plan is designed for a beginner aiming to lose weight and burn fat, focusing on full-body resistance training and cardio. By incorporating three workout sessions and rest days, the plan allows adequate recovery while maintaining a balance between strength training and cardiovascular activities.


In [18]:
# Cell 16 — Display workout plan as table
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

workout_rows = []
for day in DAYS:
    day_data = workout_plan["days"].get(day, {})
    w_type   = day_data.get("type", "Rest")
    focus    = day_data.get("focus", "-")
    duration = day_data.get("duration_min", 0)
    exercises = day_data.get("exercises", [])

    if not exercises:
        workout_rows.append({
            "Day": day, "Type": w_type, "Focus": focus,
            "Duration (min)": "-", "Exercise": "😴 Rest day",
            "Sets": "-", "Reps / Duration": "-", "Rest (sec)": "-",
        })
    else:
        for i, ex in enumerate(exercises):
            workout_rows.append({
                "Day":             day if i == 0 else "",
                "Type":            w_type if i == 0 else "",
                "Focus":           focus if i == 0 else "",
                "Duration (min)": duration if i == 0 else "",
                "Exercise":        ex.get("name", ""),
                "Sets":            ex.get("sets", "-"),
                "Reps / Duration": ex.get("reps_or_duration", "-"),
                "Rest (sec)":      ex.get("rest_sec", "-"),
            })

df_workout = pd.DataFrame(workout_rows)
print(tabulate(df_workout, headers="keys", tablefmt="rounded_outline", showindex=False))

╭───────────┬─────────┬────────────────────────────────┬──────────────────┬───────────────────────────┬────────┬───────────────────┬──────────────╮
│ Day       │ Type    │ Focus                          │ Duration (min)   │ Exercise                  │ Sets   │ Reps / Duration   │ Rest (sec)   │
├───────────┼─────────┼────────────────────────────────┼──────────────────┼───────────────────────────┼────────┼───────────────────┼──────────────┤
│ Monday    │ Workout │ Full Body Strength             │ 60               │ Squats                    │ 3      │ 12 reps           │ 60           │
│           │         │                                │                  │ Bench Press               │ 3      │ 12 reps           │ 60           │
│           │         │                                │                  │ Lat Pulldowns             │ 3      │ 12 reps           │ 60           │
│           │         │                                │                  │ Plank                     │ 3      │

In [19]:
# Cell 17 — Rich HTML workout cards (renders inline in Colab)
workout_html = ["""
<style>
  .wo-day       { margin: 20px 0 8px; font-size: 18px; font-weight: 700;
                  color: #1d3557; border-bottom: 2px solid #457b9d; padding-bottom: 4px; }
  .wo-meta      { font-size: 13px; color: #457b9d; margin-bottom: 10px; }
  .wo-rest      { background: #f1f3f5; border-radius: 8px; padding: 10px 14px;
                  color: #868e96; font-style: italic; display: inline-block; }
  .ex-table     { border-collapse: collapse; width: 100%; margin-bottom: 12px; font-size: 13px; }
  .ex-table th  { background: #1d3557; color: #fff; padding: 7px 10px; text-align: left; }
  .ex-table td  { padding: 6px 10px; border-bottom: 1px solid #dee2e6; vertical-align: top; }
  .ex-table tr:nth-child(even) td { background: #e8f4fd; }
  .tip          { color: #6c757d; font-size: 11px; font-style: italic; }
  h2.wo-title   { color: #1d3557; }
</style>
<h2 class='wo-title'>🏋️ Your Personalised 7-Day Workout Routine</h2>
<p style='color:#555;font-size:13px'>" + workout_plan.get('plan_summary','') + "</p>
"""]

# Rebuild cleanly without f-string nesting issues
summary_p = f"<p style='color:#555;font-size:13px'>{workout_plan.get('plan_summary','')}</p>"
workout_html = [
    "<style>",
    ".wo-day{margin:20px 0 8px;font-size:18px;font-weight:700;color:#1d3557;border-bottom:2px solid #457b9d;padding-bottom:4px}",
    ".wo-meta{font-size:13px;color:#457b9d;margin-bottom:10px}",
    ".wo-rest{background:#f1f3f5;border-radius:8px;padding:10px 14px;color:#868e96;font-style:italic;display:inline-block}",
    ".ex-table{border-collapse:collapse;width:100%;margin-bottom:12px;font-size:13px}",
    ".ex-table th{background:#1d3557;color:#fff;padding:7px 10px;text-align:left}",
    ".ex-table td{padding:6px 10px;border-bottom:1px solid #dee2e6;vertical-align:top}",
    ".ex-table tr:nth-child(even) td{background:#e8f4fd}",
    ".tip{color:#6c757d;font-size:11px;font-style:italic}",
    "h2.wo-title{color:#1d3557}",
    "</style>",
    "<h2 class='wo-title'>🏋️ Your Personalised 7-Day Workout Routine</h2>",
    summary_p,
]

for day in DAYS:
    day_data  = workout_plan["days"].get(day, {})
    w_type    = day_data.get("type", "Rest")
    focus     = day_data.get("focus", "")
    duration  = day_data.get("duration_min", 0)
    exercises = day_data.get("exercises", [])

    workout_html.append(f"<div class='wo-day'>📅 {day} — {w_type}</div>")

    if not exercises:
        workout_html.append("<div class='wo-rest'>😴 Rest day — recover, stretch, hydrate.</div>")
    else:
        workout_html.append(f"<div class='wo-meta'>🎯 Focus: {focus} &nbsp;|&nbsp; ⏱ {duration} min</div>")
        workout_html.append(
            "<table class='ex-table'>"
            "<tr><th>Exercise</th><th>Sets</th><th>Reps / Duration</th><th>Rest</th><th>Form Tip</th></tr>"
        )
        for ex in exercises:
            workout_html.append(
                f"<tr>"
                f"<td><strong>{ex.get('name','')}</strong></td>"
                f"<td>{ex.get('sets','-')}</td>"
                f"<td>{ex.get('reps_or_duration','-')}</td>"
                f"<td>{ex.get('rest_sec','-')}s</td>"
                f"<td class='tip'>{ex.get('tip','')}</td>"
                f"</tr>"
            )
        workout_html.append("</table>")

display(HTML("".join(workout_html)))

Exercise,Sets,Reps / Duration,Rest,Form Tip
Squats,3,12 reps,60s,Keep your chest up and knees in line with your toes.
Bench Press,3,12 reps,60s,Maintain a grip shoulder-width apart and lower the bar to your chest.
Lat Pulldowns,3,12 reps,60s,"Pull the bar down to your upper chest, squeezing your shoulder blades together."
Plank,3,30 sec,30s,Keep your body in a straight line from head to heels.
Exercise,Sets,Reps / Duration,Rest,Form Tip
Treadmill Walking/Jogging,1,30 min,0s,Start at a brisk walking pace and gradually increase to a light jog.
Russian Twists,3,15 reps each side,30s,Keep your back straight and twist your torso.
Bicycle Crunches,3,15 reps each side,30s,Engage your core and avoid pulling on your neck.
Mountain Climbers,3,30 sec,30s,Keep your core tight and move your knees towards your chest.
Exercise,Sets,Reps / Duration,Rest,Form Tip


In [20]:
# Cell 18 — Export workout plan to CSV and download
df_workout_export = df_workout.copy()
workout_csv_path  = "workout_plan_7day.csv"
df_workout_export.to_csv(workout_csv_path, index=False)
print(f"💾 Workout plan saved to: {workout_csv_path}")

# Also append workout sheet to the existing meal plan CSV bundle
combined_path = "full_plan_7day.csv"
with open(combined_path, "w") as f:
    f.write("=== MEAL PLAN ===\n")
    df.to_csv(f, index=False)
    f.write("\n=== WORKOUT PLAN ===\n")
    df_workout_export.to_csv(f, index=False)
print(f"💾 Combined plan saved to: {combined_path}")

try:
    from google.colab import files
    files.download(workout_csv_path)
    files.download(combined_path)
    print("⬇️  Downloads started in your browser.")
except ImportError:
    print("(Not running in Colab — files saved locally.)")

💾 Workout plan saved to: workout_plan_7day.csv
💾 Combined plan saved to: full_plan_7day.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Downloads started in your browser.
